# 03 — Enterprise Fraud Analytics & SQL Intelligence

## Business Context & Executive Overview
Phase 3 (**Enterprise Fraud Analytics & SQL Intelligence**) executes 45+ advanced analytical SQL queries against the Phase 2 Star Schema Data Warehouse.

This notebook showcases 6 core pillars:
1. **Executive KPIs**: Total transaction volume, fraud loss ratio %, average transaction sizes, and busiest operating days
2. **Temporal Trends**: Daily, weekly, monthly, and hourly night-time (00-05) vs daytime fraud exposure
3. **Customer & Account Risk**: Top 20 risky senders/receivers, transfer velocity, and repeat fraud offenders
4. **Bank & Channel Intelligence**: Cross-bank transfer matrix, payment format risk rankings, and currency flow distribution
5. **Anomaly Investigation**: Rapid repeat transfers, structuring attempts ($9,000–$9,999), and self-loop transfers
6. **Advanced Window Functions**: `RANK()`, `DENSE_RANK()`, `LAG()`, `LEAD()`, `NTILE()`, `PERCENT_RANK()`, and rolling 7-day fraud trends

```text
                  Phase 2 Data Warehouse
                            │
                            ▼
                  AnalyticsExecutor.py
                            │
    ┌───────────────────────┼───────────────────────┐
    │                       │                       │
    ▼                       ▼                       ▼
04_business_analytics.sql   05_reporting_views.sql   06_query_optimization.sql
  (45+ Analytical Queries)    (12 BI Views)           (B-Tree Indexing DDL)
    │                       │                       │
    └───────────────────────┼───────────────────────┘
                            ▼
                   AnalyticsReporter.py
                            │
                            ▼
               SQL_Analytics_Report.md
```

# Section 1: Environment & Warehouse Connection Setup

In [1]:
import sys
from pathlib import Path

from IPython.display import Markdown, display

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.analytics.executor import AnalyticsExecutor
from src.analytics.reporter import AnalyticsReporter
from src.warehouse.database import DB_ENGINE_TYPE, DOCS_DIR, SQL_DIR, WarehouseConnection

# Initialize Connection & Analytics Executor Engine
db_conn = WarehouseConnection(engine_type=DB_ENGINE_TYPE)
executor = AnalyticsExecutor(db_conn)

display(Markdown(f"**Connected Warehouse Engine**: `{DB_ENGINE_TYPE.upper()}`  \n**SQL Analytics Directory**: `{SQL_DIR}`"))

**Connected Warehouse Engine**: `DUCKDB`  
**SQL Analytics Directory**: `C:\Users\hiten\OneDrive\Documents\Fraud Detection\sql`

# Section 2: Executive KPIs & Advanced Fraud Analytics (`04_business_analytics.sql`)

In [2]:
# Execute 04_business_analytics.sql showcase queries (45 Statements)
analytics_results = executor.execute_script(SQL_DIR / "04_business_analytics.sql")

display(Markdown(f"**Successfully Executed `{len(analytics_results)}` Analytical SQL Query Statements!**"))

# Sample Display of Key Statements across Pillars
display(Markdown("### Pillar 1: Executive KPIs (Statement #1)"))
display(analytics_results[0]["dataframe"])

display(Markdown("### Pillar 2: Hourly Fraud Distribution (Statement #9)"))
display(analytics_results[8]["dataframe"])

display(Markdown("### Pillar 3: Top Risky Sender Accounts (Statement #15)"))
display(analytics_results[14]["dataframe"].head(10))

display(Markdown("### Pillar 6: Window Functions - RANK() & DENSE_RANK() (Statement #38)"))
display(analytics_results[37]["dataframe"].head(10))

**Successfully Executed `45` Analytical SQL Query Statements!**

### Pillar 1: Executive KPIs (Statement #1)

total_transactions,fraud_transactions,fraud_rate_pct,total_volume_paid,total_fraud_volume,fraud_loss_pct
i64,i64,f64,"decimal[38,2]","decimal[38,2]",f64
1000,0,0.0,2554199390.08,0.00,0.0


### Pillar 2: Hourly Fraud Distribution (Statement #9)

hour,transaction_count,fraud_count,hourly_fraud_rate_pct,total_hourly_volume
i64,i64,i64,f64,"decimal[38,2]"
0,1000,0,0.0,2554199390.08


### Pillar 3: Top Risky Sender Accounts (Statement #15)

sender_account,total_sent_tx,fraud_events,total_sent_volume
str,i64,i64,"decimal[38,2]"
"""80013AB30""",4,0,1341853585.17
"""800107510""",3,0,883776130.14
"""800148F70""",2,0,60832063.09
"""800056ED0""",2,0,32027256.39
"""8001696B0""",1,0,13701811.44
"""800106DA0""",6,0,13103227.72
"""800196940""",1,0,13004141.25
"""80014BEE0""",2,0,11721467.36
"""8001414F0""",3,0,9943207.86


### Pillar 6: Window Functions - RANK() & DENSE_RANK() (Statement #38)

transaction_id,sender_account,amount_paid,row_num,rank_num,dense_rank_num
str,str,"decimal[38,2]",i64,i64,i64
"""TX_647""","""8000FA930""",600096.08,1,1,1
"""TX_235""","""8000FA930""",32420.35,2,2,2
"""TX_649""","""8000FA930""",31972.56,3,3,3
"""TX_648""","""8000FA930""",28231.31,4,4,4
"""TX_409""","""80012DBC0""",10.58,1,1,1
"""TX_592""","""80015E880""",20.75,1,1,1
"""TX_821""","""8001D07E0""",2266.17,1,1,1
"""TX_830""","""8001D07E0""",17.68,2,2,2
"""TX_581""","""800159710""",3730.41,1,1,1


# Section 3: Reusable Reporting Views (`05_reporting_views.sql`)

In [3]:
# Create/Update 12 Reporting Views in the Warehouse
from src.warehouse.logger import SQLRunner

runner = SQLRunner(db_conn)
runner.execute_file(SQL_DIR / "05_reporting_views.sql")

# Query Reporting Views
vw_daily = executor.query("SELECT * FROM vw_daily_fraud_summary;")
vw_bank = executor.query("SELECT * FROM vw_bank_risk LIMIT 10;")
vw_account = executor.query("SELECT * FROM vw_account_risk LIMIT 10;")
vw_weekly = executor.query("SELECT * FROM vw_weekly_fraud_summary;")

display(Markdown("### Daily Fraud Summary View (`vw_daily_fraud_summary`)"))
display(vw_daily.head(10))

display(Markdown("### Bank Risk Scorecard View (`vw_bank_risk`)"))
display(vw_bank)

### Daily Fraud Summary View (`vw_daily_fraud_summary`)

transaction_date,total_transactions,fraud_cases,fraud_rate_pct,total_volume_paid,total_fraud_volume
date,i64,i64,f64,"decimal[38,2]","decimal[38,2]"
2022-09-01,1000,0,0.0,2554199390.08,0.00


### Bank Risk Scorecard View (`vw_bank_risk`)

bank_name,total_processed_tx,total_fraud_tx,bank_fraud_rate_pct,total_volume_usd
str,i64,i64,f64,"decimal[38,2]"
"""Bank_70""",31,0,0.0,5367795.35
"""Bank_531""",3,0,0.0,6338.71
"""Bank_3348""",1,0,0.0,713.81
"""Bank_32042""",1,0,0.0,0.01
"""Bank_3233""",2,0,0.0,4489.78
"""Bank_11047""",1,0,0.0,12.19
"""Bank_3225""",2,0,0.0,5343.86
"""Bank_11046""",3,0,0.0,2328690.97
"""Bank_3288""",2,0,0.0,5356.46


# Section 4: Query Optimization & Indexing Benchmarks (`06_query_optimization.sql`)

In [4]:
# Execute 06_query_optimization.sql (Applies Index DDL & Runs EXPLAIN Plans)
opt_results = executor.execute_script(SQL_DIR / "06_query_optimization.sql")

display(Markdown("### Optimization Benchmark Plan (`EXPLAIN ANALYZE`)"))
display(opt_results[1]["dataframe"])

### Optimization Benchmark Plan (`EXPLAIN ANALYZE`)

Count
i64
2751961334808


# Section 5: Dynamic Markdown Report & Snapshot Generation

In [5]:
reporter = AnalyticsReporter(output_dir=DOCS_DIR)
report_file = reporter.generate_report(analytics_results)

display(Markdown(f"**Analytics Markdown Report Generated Successfully!**  \n**Report Location**: `{report_file}`"))

**Analytics Markdown Report Generated Successfully!**  
**Report Location**: `C:\Users\hiten\OneDrive\Documents\Fraud Detection\docs\SQL_Analytics_Report.md`

# Section 6: Final Executive Readiness Summary

| Analytics Pillar | Queries Executed | Key Insight / Outcome |
| :--- | :---: | :--- |
| **1. Executive KPIs** | 7 Queries | Total volume, fraud loss ratio %, and operational SLAs |
| **2. Temporal Trends** | 7 Queries | Night-time (00-05) vs daytime fraud exposure patterns |
| **3. Customer Risk** | 8 Queries | Top 20 risky accounts & repeat fraud offender identification |
| **4. Entity Intelligence**| 8 Queries | Cross-bank transfer matrix & payment format vulnerability |
| **5. Anomaly Detection**| 7 Queries | Structuring ($9k-$9.9k) & rapid repeat transfer alerts |
| **6. Window Functions** | 8 Queries | `RANK()`, `LAG()`, `LEAD()`, `NTILE()`, and rolling 7-day trends |
| **Reporting Views** | 12 Views | Dynamic views (`vw_daily_fraud_summary`, `vw_bank_risk`) |
| **Query Optimization** | B-Tree Indexes | Composite B-Tree index DDL & EXPLAIN plan validation |